In [2]:
import sgp4

import importlib.metadata

version = importlib.metadata.version('sgp4')
print(version)
from sgp4.api import accelerated

2.27


In [3]:
import pandas as pd
import sys
import os
from pathlib import Path

from sgp4.api import accelerated


In [ ]:
from sgp4.api import Satrec

s = '1 25544U 98067A   19343.69339541  .00001764  00000-0  38792-4 0  9991'
t = '2 25544  51.6439 211.2001 0007417  17.6667  85.6398 15.50103472202482'
satellite = Satrec.twoline2rv(s, t)

jd, fr = 2458826.5, 0.8625
e, r, v = satellite.sgp4(jd, fr)
e

print(r)  # True Equator Mean Equinox position (km)
print(v)  # True Equator Mean Equinox velocity (km/s)

In [4]:
orbcomData=pd.read_csv('Orbcomm2026test.csv',skiprows=9)
orbcomData

,\outl0\strokewidth0 \strokec2 OBJECT_NAME,OBJECT_ID,EPOCH,MEAN_MOTION,ECCENTRICITY,INCLINATION,RA_OF_ASC_NODE,ARG_OF_PERICENTER,MEAN_ANOMALY,EPHEMERIS_TYPE,CLASSIFICATION_TYPE,NORAD_CAT_ID,ELEMENT_SET_NO,REV_AT_EPOCH,BSTAR,MEAN_MOTION_DOT,MEAN_MOTION_DDOT\
0,ORBCOMM FM06,1997-084G,2026-10-05T21:28:37.955136,14.477825,0.000177,45.0133,239.6601,315.5269,72.5343,0.0,U,25118.0,999.0,51043.0,0.000244,0.000008,0\
1,ORBCOMM FM04,1998-007C,2026-10-06T04:05:19.148928,14.358918,0.003912,107.9618,37.8387,134.5775,225.8569,0.0,U,25159.0,999.0,49252.0,0.000232,0.000005,0\
2,ORBCOMM FM19,1998-046C,2026-10-06T04:58:21.273888,14.396645,0.000464,44.9934,75.7636,151.4158,18.3922,0.0,U,25415.0,999.0,47277.0,0.000230,0.000006,0\
3,ORBCOMM FM20,1998-046D,2026-10-06T03:51:05.876352,14.381932,0.000175,44.9932,85.1506,149.0725,211.0226,0.0,U,25416.0,999.0,47276.0,0.000183,0.000004,0\
4,ORBCOMM FM27,1998-053G,2026-10-06T01:00:49.684608,14.393057,0.000160,45.0043,168.7532,28.9157,331.1781,0.0,U,25481.0,999.0,46620.0,0.000208,0.000005,0\
5,ORBCOMM FM32,1999-065C,2026-10-05T16:13:42.568032,14.372626,0.000785,45.0168,10.1945,242.5089,117.4961,0.0,U,25982.0,999.0,40298.0,0.000195,0.000004,0\
6,ORBCOMM FM109,2014-040A,2026-10-06T01:04:16.425120,14.589339,0.000177,47.0029,287.3240,330.9287,29.1500,0.0,U,40086.0,999.0,65053.0,0.000083,0.000002,0\
7,ORBCOMM FM110,2015-081D,2026-10-06T02:44:48.988032,14.589843,0.000305,47.0013,196.8894,238.5128,121.5459,0.0,U,41182.0,999.0,57382.0,0.000104,0.000003,0\
8,ORBCOMM FM118,2015-081E,2026-10-05T19:23:56.030496,14.588319,0.000316,47.0036,198.7113,190.7645,169.3174,0.0,U,41183.0,999.0,57388.0,0.000105,0.000003,0\
9,ORBCOMM FM112,2015-081F,2026-10-05T22:44:39.598656,14.589762,0.000263,46.9975,106.4496,350.8895,9.1942,0.0,U,41184.0,999.0,57512.0,0.000079,0.000002,0\


In [9]:
print(orbcomData.loc[1])




\outl0\strokewidth0 \strokec2 OBJECT_NAME                  ORBCOMM FM04
OBJECT_ID                                                     1998-007C
EPOCH                                        2026-10-06T04:05:19.148928
MEAN_MOTION                                                   14.358918
ECCENTRICITY                                                   0.003912
INCLINATION                                                    107.9618
RA_OF_ASC_NODE                                                  37.8387
ARG_OF_PERICENTER                                              134.5775
MEAN_ANOMALY                                                   225.8569
EPHEMERIS_TYPE                                                      0.0
CLASSIFICATION_TYPE                                                   U
NORAD_CAT_ID                                                    25159.0
ELEMENT_SET_NO                                                    999.0
REV_AT_EPOCH                                                    

In [15]:
orbcomData.rename(columns={orbcomData.columns[-1]: 'MEAN_MOTION_DDOT'}, inplace=True)
orbcomData.loc[0]


\outl0\strokewidth0 \strokec2 OBJECT_NAME                  ORBCOMM FM06
OBJECT_ID                                                     1997-084G
EPOCH                                        2026-10-05T21:28:37.955136
MEAN_MOTION                                                   14.477825
ECCENTRICITY                                                   0.000177
INCLINATION                                                     45.0133
RA_OF_ASC_NODE                                                 239.6601
ARG_OF_PERICENTER                                              315.5269
MEAN_ANOMALY                                                    72.5343
EPHEMERIS_TYPE                                                      0.0
CLASSIFICATION_TYPE                                                   U
NORAD_CAT_ID                                                    25118.0
ELEMENT_SET_NO                                                    999.0
REV_AT_EPOCH                                                    

In [48]:
from sgp4.api import WGS72OLD, WGS72, WGS84
from datetime import datetime, timezone
from sgp4.api import SatrecArray
import math



satArr =[]

for isat in range (len(orbcomData)-1):
    sat = Satrec()
    date_str = orbcomData.iloc[isat].EPOCH

    epoch = datetime(1949, 12, 31, 0, 0, tzinfo=timezone.utc)

    # 2. Parse the input string (assuming UTC matching your UT reference)
    dt = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)

    # 3. Calculate the time difference
    delta = dt- epoch

    # 4. Get the exact total days as a float
    days_since = delta.total_seconds() / 86400
    print(days_since)
    mean_motion_ddot = float(orbcomData.loc[isat].MEAN_MOTION_DDOT[:-1])
    print(orbcomData.loc[isat].INCLINATION)
    sat.sgp4init(
        WGS72,                # gravity model
        'i',                  # 'a' = old AFSPC mode, 'i' = improved mode
        int(orbcomData.iloc[isat].NORAD_CAT_ID),                # satnum: Satellite number
        days_since,           # epoch: days since 1949 December 31 00:00 UT
        orbcomData.loc[isat].BSTAR,           # bstar: drag coefficient (1/earth radii)
        math.radians(orbcomData.loc[isat].MEAN_MOTION_DOT), # ndot: ballistic coefficient (radians/minute^2)
        math.radians(mean_motion_ddot),                     # nddot: mean motion 2nd derivative (radians/minute^3)
        orbcomData.loc[isat].ECCENTRICITY,            # ecco: eccentricity
        math.radians(orbcomData.loc[isat].ARG_OF_PERICENTER),   # argpo: argument of perigee (radians 0..2pi)
        math.radians(orbcomData.loc[isat].INCLINATION),   # inclo: inclination (radians 0..pi)
        math.radians(orbcomData.loc[isat].MEAN_ANOMALY),   # mo: mean anomaly (radians 0..2pi)
        math.radians(orbcomData.loc[isat].MEAN_MOTION),  # no_kozai: mean motion (radians/minute)
        math.radians(orbcomData.loc[isat].RA_OF_ASC_NODE),    # nodeo: R.A. of ascending node (radians 0..2pi)
    )
    satArr.append(sat)
sat_array = SatrecArray(satArr)

28037.89488374
45.0133
28038.170360520002
107.9618
28038.20719067
44.9934
28038.160484679996
44.9932
28038.04224172
45.0043
28037.676187129997
45.0168
28038.04463455
47.0029
28038.114455879997
47.0013
28037.808287390002
47.0036
28037.94768054
46.9975
28038.133122119998
46.9994
28037.54498574
47.0062
28037.58007979
47.0028
28037.97137362
47.0007


In [50]:
from sgp4.api import jday
import numpy as np

jd1, fr1 = jday(2026, 10, 7, 12, 0, 0)
jd2, fr2 = jday(2026, 10, 7, 12, 0, 1)
e, r, v = sat_array.sgp4(np.array([jd1,jd2]),np.array( [fr1,fr2]))
print(e)
print(r)
print(v)

[[6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]
 [6 6]]
[[[ -752.99823596 -2408.85428651 -1265.98482953]
  [ -743.04586763 -2408.52395528 -1272.48341547]]

 [[  565.36916394  1016.38104402  2568.43502224]
  [  569.66432004  1005.77631553  2571.6300713 ]]

 [[ 1041.50930865  2244.2517914   1370.72666745]
  [ 1034.22214745  2251.38019623  1364.60803938]]

 [[-1045.26489682  2600.74916894  -430.47476557]
  [-1052.66368773  2596.40213086  -438.69465987]]

 [[-2535.79886435  1052.70222373   688.16444997]
  [-2540.95570337  1045.46692114   680.25015713]]

 [[ 2431.2305504  -1457.07597063    40.0108047 ]
  [ 2435.65110933 -1449.94353244    31.58366727]]

 [[ -181.25464222 -2532.93144023  1186.90989147]
  [ -172.52764352 -2536.90808373  1179.78134921]]

 [[-1742.78195768  1066.04240335  1913.34614754]
  [-1751.06226788  1058.0101      1910.27645069]]

 [[  -54.94103869  1919.11871277  2035.87417518]
  [  -66.84633383  1919.52570095  2035.1458789 ]]

 

In [45]:
from sgp4.api import SGP4_ERRORS
SGP4_ERRORS

{1: 'mean eccentricity is outside the range 0.0 to 1.0',
 2: 'nm is less than zero',
 3: 'perturbed eccentricity is outside the range 0.0 to 1.0',
 4: 'semilatus rectum is less than zero',
 5: '(error 5 no longer in use; it meant the satellite was underground)',
 6: 'mrt is less than 1.0 which indicates the satellite has decayed'}